# Orquestador EDA — Comercio Exterior (INE Bolivia)

Este notebook ejecuta automáticamente los notebooks de EDA individuales
(`01_eda_exportaciones.ipynb` y `02_eda_importaciones.ipynb`) sobre
**todos los archivos** de datos raw, usando [papermill](https://papermill.readthedocs.io/).

Cada ejecución genera un notebook de salida con los resultados en
`notebooks/output/`.

## Uso

```bash
# Desde la raíz del proyecto:
uv run jupyter notebook notebooks/00_orquestador_eda.ipynb
```

In [ ]:
# Parámetros del orquestador
# MAX_ROWS para importaciones (None = leer todo; usar número para limitar)
IMPORT_MAX_ROWS = 50000  # Limitar a 50k filas por defecto (archivos de ~400k filas)

## 1. Configuración

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import papermill as pm

# Asegurar que src/ es importable resolviendo la raíz del proyecto
current = Path.cwd().resolve()
candidates = [
    current,
    current.parent,
    current / "insight-bolivia",
    current.parent / "insight-bolivia",
    *current.parents,
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "extract.py").exists()),
    current,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.extract import list_raw_files

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
OUTPUT_DIR = NOTEBOOKS_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

EXPORT_NOTEBOOK = str(NOTEBOOKS_DIR / "01_eda_exportaciones.ipynb")
IMPORT_NOTEBOOK = str(NOTEBOOKS_DIR / "02_eda_importaciones.ipynb")

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio de salida: {OUTPUT_DIR}")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. Descubrir Archivos de Datos

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "comercio exterior"

export_files = list_raw_files(DATA_DIR / "exportaciones")
import_files = list_raw_files(DATA_DIR / "importaciones")

print(f"Archivos de exportaciones ({len(export_files)}):")
for f in export_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

print(f"\nArchivos de importaciones ({len(import_files)}):")
for f in import_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

## 3. Ejecutar EDA de Exportaciones

In [ ]:
export_results = []

for filepath in export_files:
    output_name = f"eda_export_{filepath.stem.lower().replace(' ', '_')}.ipynb"
    output_path = str(OUTPUT_DIR / output_name)

    print(f"\n{'=' * 60}")
    print(f"Ejecutando EDA: {filepath.name}")
    print(f"Salida: {output_name}")
    print(f"{'=' * 60}")

    try:
        pm.execute_notebook(
            EXPORT_NOTEBOOK,
            output_path,
            parameters={"FILE_PATH": str(filepath)},
            kernel_name="python3",
        )
        export_results.append({"file": filepath.name, "status": "OK", "output": output_name})
        print("  ✅ Completado")
    except Exception as e:
        export_results.append({"file": filepath.name, "status": "ERROR", "error": str(e)})
        print(f"  ❌ Error: {e}")

print(f"\nExportaciones procesadas: {len(export_results)}")

## 4. Ejecutar EDA de Importaciones

In [ ]:
import_results = []

for filepath in import_files:
    output_name = f"eda_import_{filepath.stem.lower().replace(' ', '_')}.ipynb"
    output_path = str(OUTPUT_DIR / output_name)

    print(f"\n{'=' * 60}")
    print(f"Ejecutando EDA: {filepath.name}")
    print(f"Salida: {output_name}")
    print(f"MAX_ROWS: {IMPORT_MAX_ROWS}")
    print(f"{'=' * 60}")

    try:
        pm.execute_notebook(
            IMPORT_NOTEBOOK,
            output_path,
            parameters={
                "FILE_PATH": str(filepath),
                "MAX_ROWS": IMPORT_MAX_ROWS,
            },
            kernel_name="python3",
        )
        import_results.append({"file": filepath.name, "status": "OK", "output": output_name})
        print("  ✅ Completado")
    except Exception as e:
        import_results.append({"file": filepath.name, "status": "ERROR", "error": str(e)})
        print(f"  ❌ Error: {e}")

print(f"\nImportaciones procesadas: {len(import_results)}")

## 5. Resumen de Ejecución

In [ ]:
import pandas as pd

print("=" * 60)
print("RESUMEN DE EJECUCIÓN")
print("=" * 60)

all_results = (
    [{"tipo": "Exportación", **r} for r in export_results]
    + [{"tipo": "Importación", **r} for r in import_results]
)
df_results = pd.DataFrame(all_results)
print(df_results[["tipo", "file", "status"]].to_string(index=False))

ok_count = sum(1 for r in all_results if r["status"] == "OK")
err_count = sum(1 for r in all_results if r["status"] == "ERROR")
print(f"\nTotal: {ok_count} exitosos, {err_count} con errores de {len(all_results)} archivos")
print(f"\nNotebooks de salida en: {OUTPUT_DIR}")